# Dynamic Volatility Modeling for European Option Hedging

This notebook demonstrates the complete pipeline on synthetic data. Replace the synthetic panel with real adjusted-close and volume data for the final empirical analysis.

In [ ]:
import numpy as np
import pandas as pd

from option_hedging.backtesting import generate_option_episodes, run_strategy_comparison
from option_hedging.evaluation import summarize_hedging_performance, volatility_forecast_metrics
from option_hedging.models import RidgeVolatilityForecaster, RollingGaussianVolatility
from option_hedging.signals import SignalLibrary, create_future_realized_volatility_target

## 1. Generate a reproducible market sample

In [ ]:
rng = np.random.default_rng(7)
periods = 900
dates = pd.bdate_range('2022-01-03', periods=periods)
regime = np.where((np.arange(periods) // 120) % 2 == 0, 0.010, 0.025)
log_returns = 0.0002 + regime * rng.standard_normal(periods)
close = 100 * np.exp(np.cumsum(log_returns))
volume = 2_000_000 * (1 + 8 * regime) * np.exp(0.15 * rng.standard_normal(periods))
market = pd.DataFrame({'date': dates, 'asset': 'SYNTH', 'close': close, 'volume': volume})
market.head()

## 2. Build signals and the future realized-volatility target

In [ ]:
features = SignalLibrary.default().compute(market)
target = create_future_realized_volatility_target(market, horizon=20)
dataset = features.merge(target, on=['date', 'asset'], validate='one_to_one').set_index('date')
target_col = 'future_realized_vol_20d'
feature_cols = [c for c in dataset.columns if c not in {'asset', target_col}]
dataset = dataset.dropna(subset=feature_cols + [target_col])
dataset.shape

## 3. Fit the signal-based volatility model chronologically

In [ ]:
split = int(0.70 * len(dataset))
train, test = dataset.iloc[:split], dataset.iloc[split:]
model = RidgeVolatilityForecaster(alpha=5.0).fit(train[feature_cols], train[target_col])
signal_forecast = model.predict(test[feature_cols])
volatility_forecast_metrics(test[target_col], signal_forecast)

## 4. Compare fixed, rolling, and signal-based hedges

In [ ]:
prices = market.set_index('date')['close']
returns = np.log(prices / prices.shift(1)).dropna()
rolling_vol = RollingGaussianVolatility(window=21).transform(returns)
test_prices = prices.reindex(test.index)
episodes = generate_option_episodes(test_prices, maturity_days=20, start_step=20)
strategies = {
    'fixed_at_start': lambda e: float(rolling_vol.loc[e.start_date]),
    'rolling_gaussian': lambda e: rolling_vol.reindex(e.prices.index[:-1]),
    'signal_ridge': lambda e: signal_forecast.reindex(e.prices.index[:-1]),
}
episode_results, hedge_paths = run_strategy_comparison(
    episodes, strategies=strategies, pricing_strategy='fixed_at_start',
    rate=0.03, transaction_cost_rate=0.0005,
)
summarize_hedging_performance(episode_results)

## Next step

Replace the synthetic panel with the final market dataset, document the data source and sample period, and repeat the same chronological experiment.